# 2차 챌린지 · MP-SENet으로 내 목소리를 구해줘!

예제 CNN(검증 PESQ-WB 1.69)을 다음과 같이 확장했습니다. 자세한 설명은 저장소의 `README.md`에 있습니다.

| 바꾼 부분 | 내용 | 노린 효과 |
| :--- | :--- | :--- |
| **모델** | MP-SENet: Dense 인코더 + 시간·주파수 Conformer 4블록 + 크기/위상 디코더 (2.05M 파라미터) | 넓은 시간·주파수 문맥, 위상까지 복원 |
| **위상** | noisy 위상을 그대로 쓰지 않고 clean 위상을 직접 추정 | 저 SNR에서 위상 왜곡으로 생기는 거친 소리 감소 |
| **손실** | 크기·위상·복소·파형·STFT 일관성 + **PESQ 판별기(MetricGAN)** | 평가 지표인 PESQ-WB를 간접적으로 직접 최적화 |
| **데이터** | **동적 혼합**: noisy − clean으로 잡음을 분리해 매번 다른 화자·잡음·SNR로 새 혼합음 생성 | 1,296쌍의 작은 데이터에서 과적합 감소 |
| **가중치 선택** | 매 epoch 일반 가중치와 **EMA** 가중치 모두 검증 164개 PESQ-WB로 평가, 최고값 저장 | 안정적인 최고 가중치 |

**실행 순서** · 1절(환경) → 2절(데이터 확인) → 3절(모델 구조) → **4절(학습)** → 5절(검증 전체 평가) → 6절(비교 청취) → 7절(제출 점검).
학습은 GPU가 필요합니다 (Colab: 런타임 → 런타임 유형 변경 → T4 GPU 이상). 학습이 끝난 뒤에는 4절을 건너뛰고 5–7절만 실행해도 됩니다.

## 1. 실행 환경과 경로

- **Colab**: 배포 데이터 폴더(`deep_learning_noise_reduction_challenge`)와 이 저장소 폴더(`se_challenge`)를 Google Drive **내 드라이브**에 두고 이 노트북을 엽니다. 다른 위치라면 아래 `DRIVE_*` 경로를 고치세요.
- **VS Code**: 저장소 폴더를 열고 이 노트북을 실행합니다. 데이터 폴더가 다른 곳에 있으면 `LOCAL_DATA_FOLDER`에 경로를 적으세요.
- 학습 결과(`last.pt`, `best_generator.pt`, 학습 기록)는 `RUN_DIR`에 저장됩니다. Colab에서는 Drive 안에 두어야 런타임이 끊겨도 이어서 학습할 수 있습니다.

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path

if sys.platform == "win32" and (Path(sys.prefix) / "conda-meta").is_dir():
    conda_bin = str(Path(sys.prefix) / "Library" / "bin")
    os.environ["PATH"] = conda_bin + os.pathsep + os.environ.get("PATH", "")
    conda_dll_handle = os.add_dll_directory(conda_bin)

packages = {"torch": "torch>=2.1", "numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
            "soundfile": "soundfile", "matplotlib": "matplotlib", "tqdm": "tqdm"}
missing = [pkg for module, pkg in packages.items() if importlib.util.find_spec(module) is None]
pip = [sys.executable, "-m", "pip", "install", "-q"]
if missing:
    subprocess.run(pip + missing, check=True)
if importlib.util.find_spec("pesq") is None:
    subprocess.run(pip + ["setuptools", "wheel", "Cython"], check=True)
    subprocess.run(pip + ["--no-build-isolation", "pesq==0.0.4"], check=True)

In [ ]:
import json
import shutil
from html import escape

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Audio, HTML, display
from scipy import signal
from scipy.ndimage import uniform_filter1d

DATA_FOLDER_NAME = "deep_learning_noise_reduction_challenge"
CODE_FOLDER_NAME = "se_challenge"
LOCAL_DATA_FOLDER = None     # 예: Path("D:/courses/deep_learning_noise_reduction_challenge")
LOCAL_CODE_FOLDER = None     # 예: Path("D:/courses/se_challenge")
DRIVE = Path("/content/drive/MyDrive")

if "google.colab" in sys.modules and not DRIVE.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

def first_existing(candidates, marker):
    return next((Path(c) for c in candidates if c is not None and (Path(c) / marker).exists()), None)

CODE_DIR = first_existing([LOCAL_CODE_FOLDER, Path.cwd(), Path.cwd() / CODE_FOLDER_NAME, DRIVE / CODE_FOLDER_NAME],
                          "model.py")
DATA_ROOT = first_existing([LOCAL_DATA_FOLDER, Path.cwd(), Path.cwd() / DATA_FOLDER_NAME,
                            Path.cwd().parent / DATA_FOLDER_NAME, DRIVE / DATA_FOLDER_NAME],
                           "validation/samples.csv")
if CODE_DIR is None:
    raise FileNotFoundError("model.py가 있는 저장소 폴더를 LOCAL_CODE_FOLDER에 지정하세요.")
if DATA_ROOT is None:
    raise FileNotFoundError("train/, validation/이 있는 배포 폴더를 LOCAL_DATA_FOLDER에 지정하세요.")
sys.path.insert(0, str(CODE_DIR))

from data import TrainDataset, extract_noise, load_split          # noqa: E402
from evaluate import evaluate                                      # noqa: E402
from infer import enhance_wave                                     # noqa: E402
from metrics import PesqPool                                       # noqa: E402
from model import PRESETS, ModelConfig, MPSENet, count_parameters, load_generator  # noqa: E402

SAMPLE_RATE = 16_000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu = torch.cuda.get_device_name(0) if device.type == "cuda" else "없음 (학습에는 GPU가 필요합니다)"
print(f"PyTorch {torch.__version__} | GPU {gpu}")
print(f"코드 {CODE_DIR}\n데이터 {DATA_ROOT}")

### 1.1. 실험 설정

학습 설정은 이 셀에서 바꿉니다. **이어 학습할 때는 같은 `RUN_NAME`과 설정을 그대로 두세요** (학습률 스케줄이 `EPOCHS` 기준입니다).
설정을 바꿔 새 실험을 하려면 `RUN_NAME`을 바꾸면 됩니다.

In [ ]:
PRESET = "full"          # "full" (2.05M, 권장) | "lite" (빠른 실험용)
EPOCHS = 150             # 최대 epoch (검증 PESQ가 PATIENCE epoch 동안 오르지 않으면 조기 종료)
BATCH_SIZE = 4
PATIENCE = 40
MAX_HOURS = 3.5          # 한 번 실행에서 학습할 최대 시간. 끝나면 저장되고, 다시 실행하면 이어서 학습합니다.
SEED = 42
EXTRA_ARGS = []          # 예: ["--dm-prob", "0.5"], ["--batch-size", "2"]

RUN_NAME = f"mpsenet_{PRESET}_seed{SEED}"
RUN_DIR = CODE_DIR / "runs" / RUN_NAME
SUBMIT_CHECKPOINT = CODE_DIR / "checkpoints" / "best_generator.pt"
PREVIEW_INDEX = 138      # 6절에서 들어볼 검증 샘플 (0부터 시작하는 행 번호)
print("학습 결과 폴더:", RUN_DIR)

## 2. 데이터와 잡음 분리 확인

배포 노트북과 같은 파형 캐시(`.waveform_cache/`)를 그대로 사용합니다. 아래 셀은 학습 쌍마다 **noisy = clean + noise** 관계가 성립하는지 확인합니다.
혼합 게인 중앙값이 1에 가깝고, 추정 SNR 범위가 `samples.csv`의 목표 SNR 범위와 비슷하면 동적 혼합을 그대로 쓸 수 있습니다.

In [ ]:
train_manifest, train_waves = load_split(DATA_ROOT, "train")
val_manifest, val_waves = load_split(DATA_ROOT, "validation")
PREVIEW_INDEX = min(PREVIEW_INDEX, len(val_waves) - 1)
_, estimated_snr = extract_noise(train_waves)
check = pd.DataFrame({"목표 SNR (dB)": train_manifest["target_snr_db"], "추정 SNR (dB)": estimated_snr})
display(check.groupby("목표 SNR (dB)")["추정 SNR (dB)"].describe().round(2))
display(train_manifest["noise_category"].value_counts().rename("학습 파일 수").to_frame().T)

In [ ]:
def comparison_stft(wave):
    wave = np.asarray(wave, dtype=np.float64)
    if wave.size < 512:
        wave = np.pad(wave, (0, 512 - wave.size))
    return signal.stft(wave, fs=SAMPLE_RATE, window="hann", nperseg=512,
                       noverlap=512 - 128, nfft=512, boundary="zeros", padded=True)


def show_comparison(clips, title, noisy_reference=None):
    # 배포 노트북과 같은 방식: 모든 패널에 같은 색상 범위와 재생 게인을 씁니다.
    if noisy_reference is None:
        noisy_reference = next(iter(clips.values()))
    reference_magnitude = max(np.abs(comparison_stft(noisy_reference)[2]).max(), 1e-12)
    columns = min(2, len(clips))
    rows = (len(clips) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(12, 3.5 * rows), sharex=True, sharey=True,
                             squeeze=False, constrained_layout=True)
    for ax, (name, wave) in zip(axes.flat, clips.items()):
        frequencies, times, spectrum = comparison_stft(wave)
        magnitude_db = 20 * np.log10(np.maximum(np.abs(spectrum), 1e-12) / reference_magnitude)
        mesh = ax.pcolormesh(times, frequencies / 1000, magnitude_db, shading="auto",
                             cmap="magma", vmin=-80, vmax=0)
        ax.set(title=name, xlabel="Time (s)", xlim=(0, len(wave) / SAMPLE_RATE), ylim=(0, SAMPLE_RATE / 2000))
    for ax in axes[:, 0]:
        ax.set_ylabel("Frequency (kHz)")
    for ax in list(axes.flat)[len(clips):]:
        ax.set_visible(False)
    fig.colorbar(mesh, ax=list(axes.flat)[:len(clips)], label="Magnitude (dB relative to noisy STFT peak)")
    fig.suptitle(title)
    plt.show()
    peak = max(np.max(np.abs(w)) for w in clips.values())
    gain = 0.99 / max(peak, 1e-8)
    players = [f'<td style="text-align:center;padding:10px"><b>{escape(n)}</b><br>'
               f'{Audio(w * gain, rate=SAMPLE_RATE, normalize=False)._repr_html_()}</td>' for n, w in clips.items()]
    rows_html = ["<tr>" + "".join(players[s:s + columns]) + "</tr>" for s in range(0, len(players), columns)]
    display(HTML('<table style="width:100%">' + "".join(rows_html) + "</table>"))


# 동적 혼합 예: 첫 학습 음성(clean)에 다른 파일에서 분리한 잡음을 새 SNR로 섞습니다.
demo_set = TrainDataset(train_waves, segment=train_waves.shape[-1], dm_prob=1.0, seed=SEED)
demo_noisy, demo_clean = demo_set[0]
show_comparison({"Original noisy": train_waves[0, 0], "Clean": train_waves[0, 1],
                 "Dynamic mix (new noise / SNR)": demo_noisy.numpy()},
                "Training pair and a dynamically mixed example")

## 3. 모델 구조

MP-SENet은 STFT의 **압축 크기 |Y|^0.3**과 **위상 ∠Y**를 2채널 영상 [시간 × 주파수]로 받습니다.

1. **Dense 인코더** – 시간축 dilation 1·2·4·8의 dense 연결 2D 합성곱 후, 주파수축을 절반(201 → 100)으로 줄입니다.
2. **TS-Conformer × 4** – 각 블록은 시간축 Conformer(주파수 빈마다 시간 방향 어텐션)와 주파수축 Conformer(프레임마다 주파수 방향 어텐션)를 차례로 적용합니다.
3. **크기 디코더** – 학습 가능한 시그모이드(상한 2)로 마스크를 만들어 noisy 크기에 곱합니다.
4. **위상 디코더** – 두 출력(실수·허수 성분)의 atan2로 위상을 **직접** 추정합니다. 예제 CNN처럼 noisy 위상을 재사용하지 않습니다.
5. **iSTFT** – 복원 크기와 위상으로 파형을 만듭니다. 입력 RMS로 정규화해 처리한 뒤 원래 크기로 되돌립니다.

In [ ]:
cfg = ModelConfig(**PRESETS[PRESET])
inspection = MPSENet(cfg).eval()
print(cfg)
rows = [{"모듈": name, "파라미터 수": count_parameters(module)} for name, module in
        [("encoder", inspection.encoder), *[(f"blocks.{i}", b) for i, b in enumerate(inspection.blocks)],
         ("mask_decoder", inspection.mask_decoder), ("phase_decoder", inspection.phase_decoder)]]
display(pd.DataFrame(rows))
print(f"전체 파라미터 {count_parameters(inspection):,}")
with torch.inference_mode():
    out = inspection.enhance(torch.from_numpy(val_waves[0, 0]).unsqueeze(0))
print(f"입력 {val_waves[0, 0].shape} → 출력 {tuple(out.shape[1:])} (같은 표본 수)")
del inspection

## 4. ▶ 학습

아래 셀은 `train.py`를 실행하고 로그를 그대로 보여 줍니다. 매 epoch 검증 164개 전체의 PESQ-WB를 일반 가중치와 EMA 가중치로 계산하고,
최고점이 나오면 `RUN_DIR/best_generator.pt`에 저장합니다(★ 표시).

- **런타임이 끊기거나 `MAX_HOURS`에 도달하면** 이 셀을 다시 실행하세요. `last.pt`에서 이어서 학습합니다.
- 첫 epoch의 소요 시간을 보고 전체 시간을 가늠하세요. GPU 메모리가 부족하면 자동으로 gradient checkpointing을 켭니다.

In [ ]:
import time

command = [sys.executable, "-u", str(CODE_DIR / "train.py"), "--data-root", str(DATA_ROOT),
           "--out-dir", str(RUN_DIR), "--preset", PRESET, "--epochs", str(EPOCHS),
           "--batch-size", str(BATCH_SIZE), "--patience", str(PATIENCE), "--seed", str(SEED),
           "--max-hours", str(MAX_HOURS), *EXTRA_ARGS]
RUN_DIR.mkdir(parents=True, exist_ok=True)
log_path = RUN_DIR / "train_log.txt"          # 학습 로그는 파일로도 남습니다 (README 기록용)
print(" ".join(command), flush=True)
start = log_path.stat().st_size if log_path.exists() else 0
with open(log_path, "a", encoding="utf-8") as log:
    process = subprocess.Popen(command, cwd=CODE_DIR, stdout=log, stderr=subprocess.STDOUT)
try:
    with open(log_path, encoding="utf-8", errors="replace") as reader:
        reader.seek(start)
        while True:
            finished = process.poll() is not None
            text = reader.read()
            if text:
                print(text, end="", flush=True)
            if finished:
                break
            time.sleep(2)
except KeyboardInterrupt:                      # 셀 중지 버튼: 학습 프로세스도 함께 멈춥니다.
    process.terminate()
    process.wait()
    print("\n중단했습니다. 이 셀을 다시 실행하면 마지막으로 저장된 epoch부터 이어서 학습합니다.")
else:
    if process.returncode != 0:
        hint = " (메모리 부족으로 강제 종료되었을 수 있습니다)" if process.returncode in (-9, 137) else ""
        raise RuntimeError(f"학습이 오류 코드 {process.returncode}로 끝났습니다{hint}. 위 로그를 확인하세요.")

### 4.1. 학습 곡선

검증 PESQ-WB(일반/EMA), 학습 중 배치의 PESQ, 크기·위상 손실, 학습률을 확인합니다.

In [ ]:
history = pd.read_csv(RUN_DIR / "training_history.csv")
best = history.loc[history[["val_pesq_wb_raw", "val_pesq_wb_ema"]].max(axis=1).idxmax()]
fig, axes = plt.subplots(1, 4, figsize=(18, 3.6), constrained_layout=True)
axes[0].plot(history.epoch, history.val_pesq_wb_raw, label="Validation (raw)")
axes[0].plot(history.epoch, history.val_pesq_wb_ema, label="Validation (EMA)")
if "train_train_pesq" in history:
    axes[0].plot(history.epoch, history.train_train_pesq, label="Train batches", alpha=0.6)
axes[0].axhline(1.6922, color="gray", ls="--", label="Example CNN")
axes[0].scatter([best.epoch], [max(best.val_pesq_wb_raw, best.val_pesq_wb_ema)], color="tab:red", zorder=3,
                label=f"Best epoch {int(best.epoch)}")
axes[0].set_ylabel("PESQ-WB"); axes[0].legend(fontsize=8)
axes[1].plot(history.epoch, history.train_mag); axes[1].set_ylabel("Magnitude MSE (train)")
axes[2].plot(history.epoch, history.train_pha); axes[2].set_ylabel("Phase loss (train)")
axes[3].plot(history.epoch, history.lr); axes[3].set(ylabel="Learning rate", yscale="log")
for ax in axes:
    ax.set_xlabel("Epoch"); ax.grid(alpha=0.25)
plt.show()
summary_path = RUN_DIR / "training_summary.json"
if summary_path.is_file():
    print(json.dumps({k: v for k, v in json.loads(summary_path.read_text(encoding="utf-8")).items()
                      if k in ("stop_reason", "last_epoch", "best_epoch", "best_validation_pesq_wb")},
                     ensure_ascii=False, indent=1))

## 5. 검증 164개 전체 평가

선택된 가중치로 **제출용 추론과 같은 함수(`infer.enhance_wave`)**를 사용해 검증 전체를 처리하고, 파일별·잡음 종류별 PESQ-WB와 SNR을 계산합니다.
결과는 `RUN_DIR/validation_per_file.csv`, `validation_by_noise.csv`에 저장됩니다.

In [ ]:
model, state = load_generator(RUN_DIR / "best_generator.pt", device)
print(f"선택된 가중치: epoch {state['epoch']} ({state['weights']}), 학습 중 검증 PESQ-WB {state['val_pesq_wb']:.4f}")
pool = PesqPool()
detail, val_enhanced = evaluate(model, val_manifest, val_waves, device, pool)
pool.close()
detail.to_csv(RUN_DIR / "validation_per_file.csv", index=False, encoding="utf-8-sig")
by_noise = (detail.groupby("noise_category")[["input_pesq_wb", "model_pesq_wb", "delta_pesq_wb", "delta_snr_db"]]
            .mean().sort_values("model_pesq_wb"))
by_noise.to_csv(RUN_DIR / "validation_by_noise.csv", encoding="utf-8-sig")
display(pd.DataFrame({
    "비교": ["잡음 입력", "MP-SENet 복원", "개선량"], "파일 수": len(detail),
    "평균 PESQ-WB": [detail.input_pesq_wb.mean(), detail.model_pesq_wb.mean(), detail.delta_pesq_wb.mean()],
    "평균 SNR (dB)": [detail.input_snr_db.mean(), detail.model_snr_db.mean(), detail.delta_snr_db.mean()],
}).round(4))
display(by_noise.round(3))
if "target_snr_db" in detail:
    display(detail.groupby("target_snr_db")[["input_pesq_wb", "model_pesq_wb"]].mean().round(3))
print("개선량이 가장 작은 파일 5개 (6절 PREVIEW_INDEX로 들어보세요)")
display(detail.nsmallest(5, "delta_pesq_wb")[["segment_id", "noise_category", "target_snr_db",
                                               "input_pesq_wb", "model_pesq_wb"]])

## 6. 같은 입력에서 Noisy · Clean · Wiener · 예제 CNN · MP-SENet 비교

1차 예제의 Wiener 필터와 배포된 예제 CNN 가중치(`outputs/best_mask_cnn.pt`가 있을 때)를 같은 샘플에 적용해 비교합니다.
`PREVIEW_INDEX`를 바꿔 다른 샘플도 들어 보세요. 모든 스펙트로그램은 같은 색상 범위와 재생 게인을 씁니다.

In [ ]:
def wiener_filter(noisy):
    # 배포 노트북(1차 예제)과 같은 Wiener 필터
    noisy = np.asarray(noisy, dtype=np.float64)
    _, times, spectrum = comparison_stft(noisy)
    power = np.abs(spectrum) ** 2
    centers = np.rint(times * SAMPLE_RATE).astype(int)
    full = np.flatnonzero((centers >= 256) & (centers + 256 <= len(noisy)))
    if not len(full):
        full = np.arange(power.shape[1])
    count = max(1, int(np.ceil(0.20 * len(full))))
    selected = full[np.argsort(power[:, full].mean(axis=0), kind="stable")[:count]]
    noise = np.maximum(uniform_filter1d(power[:, selected].mean(axis=1), size=3, mode="nearest"), 1e-18)
    posterior_snr = power / noise[:, None]
    enhanced = np.empty_like(spectrum)
    previous = np.zeros_like(noise)
    for frame in range(spectrum.shape[1]):
        instantaneous = np.maximum(posterior_snr[:, frame] - 1.0, 0.0)
        prior = instantaneous if frame == 0 else 0.98 * previous / noise + 0.02 * instantaneous
        gain = np.clip(prior / (1 + prior), 0.05, 1)
        enhanced[:, frame] = gain * spectrum[:, frame]
        previous = np.abs(enhanced[:, frame]) ** 2
    _, restored = signal.istft(enhanced, fs=SAMPLE_RATE, window="hann", nperseg=512,
                              noverlap=512 - 128, nfft=512, input_onesided=True, boundary=True)
    return restored[:len(noisy)]


class MaskCNN(torch.nn.Module):
    # 배포 예제 CNN (기본 설정). 비교용으로만 사용합니다.
    def __init__(self):
        super().__init__()
        nn = torch.nn
        self.mask_net = nn.Sequential(nn.Conv1d(257, 64, 5, padding=2), nn.ReLU(), nn.Conv1d(64, 64, 3, padding=1),
                                      nn.ReLU(), nn.Conv1d(64, 257, 1), nn.Sigmoid())
        self.register_buffer("window", torch.hann_window(512))

    def enhance(self, noisy):
        spec = torch.stft(noisy, n_fft=512, hop_length=128, window=self.window, return_complex=True)
        out = self.mask_net(torch.log1p(spec.abs())) * spec
        return torch.istft(out, n_fft=512, hop_length=128, window=self.window, length=noisy.shape[-1])


example_cnn = None
cnn_path = DATA_ROOT / "outputs" / "best_mask_cnn.pt"
if cnn_path.is_file():
    try:
        example_cnn = MaskCNN()
        example_cnn.load_state_dict(torch.load(cnn_path, map_location="cpu", weights_only=True))
        example_cnn.eval()
    except Exception as error:   # 예제 설정을 바꿔 학습했다면 구조가 달라 불러오지 못할 수 있습니다.
        print("예제 CNN 가중치를 불러오지 못했습니다:", error)
        example_cnn = None

In [ ]:
from metrics import pesq_safe

example = val_manifest.iloc[PREVIEW_INDEX]
noisy, clean = val_waves[PREVIEW_INDEX]
clips = {"Noisy input": noisy, "Clean reference": clean, "Wiener filter": wiener_filter(noisy)}
if example_cnn is not None:
    with torch.inference_mode():
        clips["Example CNN"] = example_cnn.enhance(torch.from_numpy(noisy).unsqueeze(0))[0].numpy()
clips["MP-SENet"] = val_enhanced[PREVIEW_INDEX]
print(f"샘플 {PREVIEW_INDEX} | {example.segment_id} | {example.noise_category} | 목표 SNR {example.target_snr_db:g} dB")
display(pd.DataFrame({"PESQ-WB": {name: pesq_safe(clean, wave) for name, wave in clips.items()
                                  if name != "Clean reference"}}).T.round(3))
show_comparison(clips, "Noisy / Clean / Wiener / Example CNN / MP-SENet", noisy_reference=noisy)

## 7. 제출 전 점검: 검증셋을 테스트셋처럼 처리하기

조교 평가와 같은 흐름으로 확인합니다.

1. 검증 noisy WAV 원본을 `val_test/input/`에 복사합니다 (clean은 `val_test/clean/`).
2. **제출용 `infer.py`**로 `val_test/output/`을 만듭니다.
3. `check_submission.py`로 파일 누락·추가 WAV·표본률·채널·표본 수를 점검하고, 저장된 WAV로 PESQ-WB 평균을 다시 계산합니다.
4. 문제가 없으면 선택된 가중치를 `checkpoints/best_generator.pt`로 복사합니다. 이 파일을 GitHub에 올립니다.

In [ ]:
VAL_TEST = CODE_DIR / "val_test"
shutil.rmtree(VAL_TEST / "output", ignore_errors=True)
SUBMIT_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(RUN_DIR / "best_generator.pt", SUBMIT_CHECKPOINT)
steps = [
    ["check_submission.py", "export", "--data-root", str(DATA_ROOT), "--dest", str(VAL_TEST)],
    ["infer.py", "--input", str(VAL_TEST / "input"), "--output", str(VAL_TEST / "output"),
     "--checkpoint", str(SUBMIT_CHECKPOINT)],
    ["check_submission.py", "check", "--input", str(VAL_TEST / "input"), "--output", str(VAL_TEST / "output"),
     "--clean", str(VAL_TEST / "clean")],
]
for step in steps:
    result = subprocess.run([sys.executable, *step], cwd=CODE_DIR, capture_output=True, text=True,
                            encoding="utf-8", errors="replace")
    print(result.stdout[-2000:], result.stderr[-2000:] if result.returncode else "")
    if result.returncode:
        raise RuntimeError(f"실패: {' '.join(step)}")
print(f"제출 가중치: {SUBMIT_CHECKPOINT} ({SUBMIT_CHECKPOINT.stat().st_size / 2**20:.1f} MiB)")
print("README의 검증 PESQ-WB 표와 학습 설정을 갱신한 뒤 커밋하세요.")